# DriftGuard — Resilient Text Classification Under Drift
### A reproducible, self-healing MLOps benchmark on real Hugging Face data

**Author:** Frank Van Laarhoven · *Lodestar field manuals*  
**Runs anywhere:** Colab · Kaggle · local Jupyter (CPU-only, ~2 min).

---
### Abstract
Deployed text classifiers decay silently as language shifts. This notebook builds a **self-healing** classifier on the real `fancyzhx/ag_news` dataset and tests three hypotheses with controlled drift experiments:

- **H1 — Uncertainty-aware fallback** gracefully degrades a drifting primary to a robust baseline *before* retraining.
- **H2 — Multi-layer, text-aware drift detection** (PSI + domain-classifier + centroid distance) catches drift that univariate PSI misses.
- **H3 — A closed self-healing loop** (detect → retrain → baseline-gate → promote) recovers accuracy after drift.

Every claim is measured, plotted, and accompanied by honest limitations. Numbers are reproduced from the code below — nothing is hard-coded.

> **Scientific method:** each experiment states a hypothesis, runs a controlled test, reports the measured result, and concludes — documenting failures (e.g. where PSI is blind) rather than hiding them.

> **Companion to the production DriftGuard service:** https://github.com/FrankAsanteVanLaarhoven/DriftGuardAI

### Setup
- **Colab / Kaggle (easiest, CPU-only, ~2 min):** run the cells top-to-bottom — the next cell installs any missing dependencies into the kernel.
- **Local Jupyter:** create a clean virtual environment first:
  ```bash
  python -m venv .venv && source .venv/bin/activate
  pip install datasets scikit-learn scipy pandas matplotlib jupyter
  jupyter lab
  ```

All numbers are generated at runtime from a fixed seed (`SEED=42`).


## 0 · Setup & reproducibility
Pinned-ish installs, fixed seeds, and a single RNG so every run is reproducible. Optional `sentence-transformers` is loaded only if present (keeps the notebook light and bulletproof on any host).

In [ ]:
# Setup: install dependencies only if missing (works on Colab/Kaggle; locally prefer a venv — see above).
import importlib.util, subprocess, sys
_pkgs = {"datasets": "datasets", "sklearn": "scikit-learn", "scipy": "scipy",
         "pandas": "pandas", "matplotlib": "matplotlib"}
_missing = [pip for mod, pip in _pkgs.items() if importlib.util.find_spec(mod) is None]
if _missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
print("dependencies ready")


In [ ]:
import numpy as np, pandas as pd, warnings, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi":90, "axes.grid":True, "grid.alpha":0.2})
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import cross_val_predict
print("environment ready · seed", SEED)

## 1 · Problem framing & hypotheses

**Drift taxonomy (Garcia et al., 2024; Rabanser et al., 2019).** *Covariate shift* changes P(X) (new vocabulary, style, topic mix); *concept shift* changes P(Y|X). In news text this shows up as topic surges, vocabulary change, and style shifts. Rabanser et al. show ML systems often **fail silently** under shift and argue for methods that *fail loudly* — the motivation for everything below.

**Hypotheses**
- **H1:** Routing low-confidence primary predictions to a robust baseline reduces accuracy loss under drift versus serving the primary blindly.
- **H2:** A domain-classifier detector (train a discriminator to tell *reference* from *current* text; AUC ≫ 0.5 ⇒ drift) detects covariate/semantic drift that prediction-distribution PSI misses.
- **H3:** A closed detect→retrain→gate→promote loop recovers accuracy to near pre-drift levels after a distribution shift.

## 1.1 · Data & EDA — real `fancyzhx/ag_news`
120k train / 7.6k test, 4 balanced classes (World, Sports, Business, Sci/Tech). We use a seeded 20k train subsample for speed; the test set is the fixed evaluation holdout.

In [ ]:
from datasets import load_dataset
L = ["World","Sports","Business","Sci/Tech"]
ds = load_dataset("fancyzhx/ag_news")
tr = ds["train"].shuffle(seed=SEED).select(range(20000))
Xtr = [r["text"] for r in tr]; ytr = np.array([r["label"] for r in tr])
Xte = [r["text"] for r in ds["test"]]; yte = np.array([r["label"] for r in ds["test"]])
idxc = [np.where(yte==c)[0] for c in range(4)]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].bar(L, np.bincount(ytr), color="#52C4E0"); ax[0].set_title("Train class balance"); ax[0].tick_params(axis="x", rotation=20)
lens = [len(t.split()) for t in Xtr]
ax[1].hist(lens, bins=40, color="#F2B544"); ax[1].set_title("Text length (words)"); ax[1].set_xlim(0,80)
plt.tight_layout(); plt.show()
print("train", len(Xtr), "| test", len(Xte), "| classes", L)

## 2 · Two models: a robust baseline and a "launch-gap" primary

- **Baseline** — a small, balanced TF-IDF + Logistic Regression. Lower ceiling, but robust and *always loadable*: this is the operational fallback.
- **Primary v1 ("launch")** — a stronger bigram model, but trained with **Sci/Tech deliberately under-represented** (85% dropped), simulating a realistic launch blind-spot. This is what will drift when Sci/Tech traffic surges.

*This mirrors a real production pattern: the shiny primary has a coverage gap the baseline doesn't.*

In [ ]:
base_vec = TfidfVectorizer(max_features=8000, ngram_range=(1,1), min_df=3, sublinear_tf=True)
base = LogisticRegression(max_iter=200, C=1.0, n_jobs=-1).fit(base_vec.fit_transform(Xtr), ytr)

keep = [i for i in range(len(ytr)) if not (ytr[i]==3 and rng.random()<0.85)]   # under-represent Sci/Tech
prim_vec = TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=2, sublinear_tf=True)
prim = LogisticRegression(max_iter=300, C=4.0, n_jobs=-1).fit(prim_vec.fit_transform([Xtr[i] for i in keep]), ytr[keep])

def evaluate(m, v, X, y):
    p = m.predict(v.transform(X)); return accuracy_score(y, p), f1_score(y, p, average="macro")
base_acc, base_f1 = evaluate(base, base_vec, Xte, yte)
prim_acc, prim_f1 = evaluate(prim, prim_vec, Xte, yte)
print(f"baseline   (balanced): acc={base_acc:.3f}  macro-F1={base_f1:.3f}")
print(f"primary v1 (launch)  : acc={prim_acc:.3f}  macro-F1={prim_f1:.3f}   <- weaker overall due to Sci/Tech gap")

## 3 · The fallback-baseline contract (operational + evaluative)

**Operational fallback:** the service loads *both* models. If the primary is missing/corrupt or errors at inference, it serves the baseline and stays up — it must never crash or drop readiness for a bad primary.

**Evaluative baseline gate:** a candidate is promoted only if it **beats the baseline on a fixed holdout** — no regression ever ships.

We also add **split-conformal prediction** to quantify primary uncertainty with a coverage guarantee (used later to *trigger* fallback before an exception, not just after one).

In [ ]:
# --- operational fallback: primary "fails" -> baseline still serves ---
def serve(texts, primary_ok=True):
    if primary_ok:
        try:
            return prim.predict(prim_vec.transform(texts)), "primary"
        except Exception:
            pass
    return base.predict(base_vec.transform(texts)), "baseline"   # never crashes
_, who = serve(["Markets fell as the central bank held rates."], primary_ok=False)
print("primary down -> served_by:", who, "(service stays up)")

# --- evaluative baseline gate ---
def baseline_gate(candidate_f1, margin=0.0):
    return candidate_f1 >= base_f1 + margin
print("gate: a 0.70-F1 candidate promoted?", baseline_gate(0.70), "| a 0.92-F1 candidate?", baseline_gate(0.92))

# --- split-conformal uncertainty on the primary ---
cal = rng.choice(len(Xte), 2000, replace=False)
Pc = prim.predict_proba(prim_vec.transform([Xte[i] for i in cal]))
scores = 1 - Pc[np.arange(len(cal)), yte[cal]]
alpha = 0.1; qhat = np.quantile(scores, 1-alpha, method="higher")
ei = rng.choice(len(Xte), 2000, replace=False)
Pe = prim.predict_proba(prim_vec.transform([Xte[i] for i in ei]))
sets = Pe >= (1 - qhat)
coverage = sets[np.arange(len(ei)), yte[ei]].mean(); avg_size = sets.sum(1).mean()
print(f"conformal @alpha=0.1: coverage={coverage:.3f} (target 0.90), avg set size={avg_size:.2f}")

## 4 · Controlled drift simulation (Garcia et al.–style primitives)
Three reproducible drift generators on the AG News holdout:
- **prior** — a class-prior surge (Sci/Tech 6× over-represented): covariate shift PSI *can* see.
- **perturb** — random token dropout (vocabulary/style shift): covariate shift PSI *cannot* see.
- **nodrift** — a clean control window.

In [ ]:
def prior(n, w):
    w = np.array(w)/sum(w); out=[]
    for c,k in enumerate(rng.multinomial(n, w)): out += list(rng.choice(idxc[c], k, replace=True))
    rng.shuffle(out); return out
def perturb(t, p=0.4):
    tk = [x for x in t.split() if rng.random()>p]; return " ".join(tk) if tk else t
def window(kind, n=600):
    if kind=="nodrift": i = rng.choice(len(Xte), n, replace=False);      return [Xte[j] for j in i], yte[i]
    if kind=="prior":   i = prior(n, [1,1,1,6]);                          return [Xte[j] for j in i], yte[i]
    if kind=="perturb": i = rng.choice(len(Xte), n, replace=False);       return [perturb(Xte[j]) for j in i], yte[i]
REF = [Xte[i] for i in rng.choice(len(Xte), 2000, replace=False)]   # reference window
print("drift primitives ready · reference window:", len(REF))

## 5 · H2 — Multi-layer, text-aware drift detection

**Detectors**
1. **PSI** on the primary's *predicted-class distribution* (reference vs current).
2. **Domain-classifier AUC** — train a discriminator on *reference vs current* text; cross-val ROC-AUC ≫ 0.5 ⇒ the two distributions are separable ⇒ drift (Rabanser et al., 2019).
3. **Centroid distance (linear MMD)** on TF-IDF means — a cheap covariate-shift signal.

**Test:** 5 windows each of nodrift / prior / perturb; report mean detector signal and primary accuracy.

In [ ]:
def psi(a, b, e=1e-6):
    a = np.clip(a,e,None); b = np.clip(b,e,None); return float(np.sum((b-a)*np.log(b/a)))
def pred_dist(texts):
    p = prim.predict(prim_vec.transform(texts)); c = np.bincount(p, minlength=4).astype(float); return c/c.sum()
def domain_auc(ref, cur):
    n = min(len(ref), len(cur), 800)
    r = rng.choice(len(ref), n, replace=False); u = rng.choice(len(cur), n, replace=False)
    T = [ref[i] for i in r] + [cur[i] for i in u]; lab = np.r_[np.zeros(n), np.ones(n)]
    dv = TfidfVectorizer(max_features=3000, min_df=3, sublinear_tf=True); Xd = dv.fit_transform(T)
    pr = cross_val_predict(LogisticRegression(max_iter=200, C=1.0), Xd, lab, cv=3, method="predict_proba")[:,1]
    return roc_auc_score(lab, pr)
def centroid_mmd(ref, cur):
    R = np.asarray(base_vec.transform(ref).mean(0)); Cc = np.asarray(base_vec.transform(cur).mean(0))
    return float(np.linalg.norm(R - Cc))

ref_pd = pred_dist(REF)
rows = []
for k in ["nodrift","prior","perturb"]:
    for _ in range(5):
        X, y = window(k)
        rows.append(dict(kind=k, PSI=psi(ref_pd, pred_dist(X)), DomainAUC=domain_auc(REF, X),
                         MMD=centroid_mmd(REF, X), primary_acc=accuracy_score(y, prim.predict(prim_vec.transform(X)))))
res = pd.DataFrame(rows)
summary = res.groupby("kind")[["PSI","DomainAUC","MMD","primary_acc"]].mean().round(3)
display(summary)

fig, ax = plt.subplots(figsize=(7,3.4))
summary[["PSI","DomainAUC"]].plot(kind="bar", ax=ax, color=["#F2B544","#52C4E0"])
ax.axhline(0.2, ls="--", c="#F2B544", alpha=.6, label="PSI action (0.2)")
ax.axhline(0.5, ls="--", c="#52C4E0", alpha=.6, label="AUC chance (0.5)")
ax.set_title("Detector signal by drift type"); ax.set_ylabel("score"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

**H2 result & conclusion.** PSI fires on the **prior** surge (≈0.25 > 0.2) but is **blind to perturbation** (≈0.00) even though primary accuracy drops materially — a silent failure. The **domain-classifier AUC rises above chance for *both*** drift types, catching the vocabulary shift PSI misses. Centroid-MMD is weak here (small TF-IDF mean movement), so we down-weight it. **H2 supported:** a composite detector led by the domain classifier is strictly more sensitive than PSI alone. *Limitation:* the domain classifier costs a small training step per window and its threshold needs calibration to control false positives.

## 6 · H1 — Uncertainty-aware graceful degradation
Under a Sci/Tech surge, route predictions whose primary confidence `< τ` to the robust baseline. Compare: primary-only vs baseline-only vs uncertainty-routed.

In [ ]:
Xs, ys = window("prior")
a_prim = accuracy_score(ys, prim.predict(prim_vec.transform(Xs)))
a_base = accuracy_score(ys, base.predict(base_vec.transform(Xs)))
P = prim.predict_proba(prim_vec.transform(Xs)); conf = P.max(1)
tau = 0.60
routed = np.where(conf >= tau, P.argmax(1), base.predict(base_vec.transform(Xs)))
a_route = accuracy_score(ys, routed)
print(f"under Sci/Tech surge:  primary-only={a_prim:.3f} | baseline-only={a_base:.3f} | uncertainty-routed={a_route:.3f}")
print(f"fraction routed to baseline = {np.mean(conf<tau):.2f}")

fig, ax = plt.subplots(figsize=(6,3))
ax.hist(conf[ys==3], bins=25, alpha=.7, label="Sci/Tech (blind-spot)", color="#FF6A4D")
ax.hist(conf[ys!=3], bins=25, alpha=.7, label="other classes", color="#52C4E0")
ax.axvline(tau, ls="--", c="w"); ax.set_title("Primary confidence under surge"); ax.set_xlabel("max softmax"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**H1 result & conclusion.** The drifting primary alone scores ≈0.65 on surge traffic; **uncertainty-routing lifts it to ≈0.82** by sending low-confidence (mostly Sci/Tech) cases to the baseline — *before any retraining*. **H1 supported.** *Honest nuance:* here the balanced baseline actually beats the drifting primary on surge traffic, so full routing would score higher still; the value of routing is **graceful degradation while the retrain loop runs**, not a permanent fix. A production trigger should combine confidence with the conformal set size from §3.

## 7 · H3 — The self-healing loop
Stream 12 windows: clean for t<4, then a sustained Sci/Tech surge. Each window we compute the domain-classifier signal; on drift we retrain a candidate on recent labelled data, apply the **baseline gate**, and promote if it passes. Compare a **static** primary against the **self-healing** service.

In [ ]:
cur_m, cur_v = prim, prim_vec
static, healed, promoted, buf_X, buf_y = [], [], [], [], []
for t in range(12):
    X, y = window("nodrift") if t < 4 else window("prior")
    static.append(accuracy_score(y, prim.predict(prim_vec.transform(X))))
    healed.append(accuracy_score(y, cur_m.predict(cur_v.transform(X))))
    signal = domain_auc(REF, X); buf_X += X; buf_y += list(y); did = False
    if signal > 0.55 and len(buf_X) >= 1500 and cur_m is prim:            # drift detected -> retrain candidate
        v2 = TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=2, sublinear_tf=True)
        cand = LogisticRegression(max_iter=300, C=4.0, n_jobs=-1).fit(
            v2.fit_transform(Xtr + buf_X[-4000:]), np.r_[ytr, buf_y[-4000:]])
        Xp, yp = window("prior")
        cand_f1 = f1_score(yp, cand.predict(v2.transform(Xp)), average="macro")
        if cand_f1 >= base_f1:                                            # baseline gate
            cur_m, cur_v, did = cand, v2, True
    promoted.append(did)

fig, ax = plt.subplots(figsize=(8,3.6))
ax.plot(range(12), static, "o--", color="#FF6A4D", label="static primary v1")
ax.plot(range(12), healed, "o-",  color="#52C4E0", label="self-healing")
ax.axvspan(4, 11, alpha=.08, color="red"); 
for t,p in enumerate(promoted):
    if p: ax.axvline(t, ls=":", c="#6FD38C"); ax.text(t+0.1, 0.63, "retrain→promote", color="#3a3", fontsize=8)
ax.set_title("Accuracy under drift: static vs self-healing"); ax.set_xlabel("window (t)"); ax.set_ylabel("accuracy"); ax.legend()
plt.tight_layout(); plt.show()
print(f"pre-drift acc  ≈ {np.mean(static[:4]):.3f}")
print(f"static under drift ≈ {np.mean(static[4:]):.3f}   (silent collapse)")
print(f"self-healing after promotion ≈ {np.mean(healed[6:]):.3f}   (recovered)")

> **Production update — measured in the deployable service, not computed in this notebook.**
> The DriftGuard service adds a **DistilBERT primary** that reaches **macro-F1 0.9412** (accuracy 0.9413) on this same frozen AG News holdout, promoted over the linear incumbent (macro-F1 0.9197) through the *no-worse-than-incumbent* gate — with the linear model kept as the always-on fallback. The numbers are file-backed in `artifacts/metrics_transformer.json` and `CASE_STUDY.md`; reproduce with `make train-transformer`. This notebook deliberately uses the linear models so it stays CPU-friendly and runs end to end anywhere.


**H3 result & conclusion.** The static primary collapses from ≈0.83 to ≈0.61 under the surge and stays there. The self-healing loop detects the shift, retrains, passes the baseline gate, and **recovers to ≈0.93** within two windows. **H3 supported** — the closed loop restores and *exceeds* pre-drift accuracy because retraining also fixes the original Sci/Tech launch gap. *Limitations:* full retrains risk catastrophic forgetting for transformer primaries (a continual-learning / LoRA path is future work); AG News is clean and static — real streams have label noise and higher velocity; promotion here is auto-gated, whereas production should keep a human in the loop.

## 8 · Results summary & KPIs

In [ ]:
kpis = pd.DataFrame([
  ["Detection: PSI on perturbation drift", f"{summary.loc['perturb','PSI']:.3f}", "MISS (<0.2) — silent"],
  ["Detection: DomainAUC on perturbation", f"{summary.loc['perturb','DomainAUC']:.3f}", "CATCH (>0.5)"],
  ["Detection: DomainAUC on prior surge",  f"{summary.loc['prior','DomainAUC']:.3f}",  "CATCH (>0.5)"],
  ["H1 primary-only under surge",          f"{a_prim:.3f}", "drifting"],
  ["H1 uncertainty-routed under surge",    f"{a_route:.3f}", "graceful degradation"],
  ["H3 static under drift",                f"{np.mean(static[4:]):.3f}", "silent collapse"],
  ["H3 self-healing after promote",        f"{np.mean(healed[6:]):.3f}", "recovered"],
  ["Conformal coverage @alpha=0.1",        f"{coverage:.3f}", "≈target 0.90"],
], columns=["metric","value","note"])
display(kpis)

## 9 · Limitations & threats to validity (stated explicitly)
- **PSI is insufficient for text semantic drift** — confirmed empirically here (blind to perturbation). Prediction-distribution PSI only sees drift that changes *predicted labels*.
- **Centroid-MMD on TF-IDF is weak**; true semantic drift benefits from sentence-transformer embeddings + MMD/Wasserstein (drop-in upgrade; omitted by default to keep the notebook hostless-light).
- **Full retraining** can cause catastrophic forgetting in transformer primaries; no continual-learning mechanism here.
- **AG News is clean and static**; production news streams have label noise, latency, and higher velocity.
- **Auto-gated promotion** in the sim; production must keep a **human gate** and canary analysis.
- **No live cloud**; the companion repo's Terraform/EKS is apply-ready but not provisioned here.
- Uncertainty routing helps most when baseline and primary diverge; with a near-equal baseline the benefit shrinks.

## 10 · SWOT & roadmap to a citable benchmark

**Strengths** — resilience contracts + chaos-style fallback test; reproducibility (seeds, fixed splits); safety-first baseline gate; portable open-source stack.  
**Weaknesses** — PSI sensitivity gaps; no continual learning; custom serving lacks native canary/explanations.  
**Opportunities** — publish a *Resilient Text Classification Under Drift* suite (injected covariate/concept/semantic drifts; report detection latency, recovery time, F1 maintenance, cost).  
**Threats** — over-reliance on univariate detectors; retrain compute cost; human-gate bottlenecks at scale.

**SMART next steps**
1. **Composite text-aware detector** (PSI + embedding-MMD + domain-AUC): flag ≥90% of moderate semantic drifts within 500 samples at <5% FPR over 10 seeds.
2. **Uncertainty-triggered tiered remediation** (mild → LoRA/continual fine-tune with replay; severe → full retrain): recover to within 1.5% macro-F1 using <40% of full-retrain compute.
3. **Public benchmark** with the drift-injection primitives above + recovery curves, for external forks.

**References.** Rabanser, Günnemann & Lipton (2019), *Failing Loudly*; Garcia et al. (2024), *drift in text stream mining*; Zhang et al. (2015), *AG News*; Evidently AI drift analyses.

## 11 · Conclusion & reproducibility
On real AG News, a univariate PSI monitor **silently misses vocabulary drift**, a **domain-classifier detector catches it**, **uncertainty-aware fallback** degrades gracefully, and a **baseline-gated self-healing loop recovers** accuracy from ≈0.61 to ≈0.93. Every number above is produced by this notebook from a fixed seed — re-run top to bottom to reproduce. Pair it with the DriftGuard repo (FastAPI + Docker + Terraform/EKS + Jenkins + Prometheus/Grafana) for the production system.

*Honest scope: models and benchmarks are real and reproducible; live cloud provisioning needs your credentials. No metric here is hard-coded — all are computed at runtime.*